In [ ]:
"""
Data Quality Assessment and Validation

This notebook checks the raw profilometer data for quality issues and identifies
samples that may require manual intervention or removal. It validates:
1. Before/after image pair completeness
2. Height profile and image shape consistency
3. Overall dataset integrity

Output: Reports on data quality issues for manual review before processing.
"""

import os
import sys
from pathlib import Path

import cv2 as cv
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches

In [ ]:
class Sample:
    """
    Container for profilometer measurement data of a corroded sample.

    Loads and parses the profilometer RGBimage and height profile from CSV files.
    Also extracts calibration metadata (pixel size, dimensions, unit).

    Attributes:
        data_path (Path): Path to the root data directory
        sample_id (str): Unique sample identifier (e.g., "x-x-x-y")
        image (np.ndarray): RGB image from profilometer (shape: [H, W, 3])
        height (np.ndarray): Height profile map (shape: [H, W])
        pixel_size (float): Pixel size in mm (converted from micrometers)
        horizontal (int): Image width in pixels
        vertical (int): Image height in pixels
        unit (str): Measurement unit from profilometer
    """
    # Conversion factor from micrometers to millimeters
    UM2MM = 1e-3

    def __init__(self, data_path: Path, time: str, sample_id: str) -> None:
        """
        Initialize Sample by loading image and height profile data.

        Args:
            data_path: Root path to processed data directory
            time: Either "before" or "after"
            sample_id: Unique sample identifier

        Raises:
            FileNotFoundError: If height CSV or image PNG file not found
        """
        self.data_path = data_path
        self.sample_id = sample_id

        # Read metadata from profilometer CSV header
        height_path = data_path / time / "height" / f"{sample_id}.csv"
        with open(height_path, 'r') as file:
            for _ in range(21):
                line = next(file).strip().replace('"', '').split(",")

                if line[0] == "XY calibration":
                    # Convert from um to mm
                    self.pixel_size = float(line[1]) * Sample.UM2MM
                if line[0] == "Horizontal":
                    self.horizontal = int(line[1])
                if line[0] == "Vertical":
                    self.vertical = int(line[1])
                if line[0] == "Unit":
                    self.unit = line[1]

        # Read height profile (skip CSV header with metadata)
        height = np.genfromtxt(height_path, skip_header=23, delimiter=",", dtype=str)
        height = np.char.strip(height, '"')
        height[height == ''] = np.nan
        self.height = height.astype(float)

        # Read RGB image from profilometer scanner
        image_path = data_path / time / "profilometer" / f"{sample_id}.png"
        self.image = cv.imread(image_path, cv.IMREAD_COLOR)

In [ ]:
def test_image_height_equality(path: Path, sample_id: str) -> tuple | None:
    """
    Check if image dimensions match height map dimensions for before and after.

    Detects misalignment between RGB images and height profiles, which can cause
    problems during sample centering and masking.

    Args:
        path: Root path to data directory
        sample_id: Sample identifier

    Returns:
        Tuple of (sample_id, before_image_shape, before_height_shape,
                 after_image_shape, after_height_shape) if mismatch detected
        None if shapes match correctly
    """
    try:
        after = Sample(path, "after", sample_id)
    except FileNotFoundError:
        return None

    before = Sample(path, "before", sample_id)

    # Check if any image-height pair has mismatched dimensions
    if not ((before.image.shape[:2] == before.height.shape[:2]) and
            (after.image.shape[:2] == after.height.shape[:2])):
        return (sample_id, before.image.shape[:2], before.height.shape[:2],
                after.image.shape[:2], after.height.shape[:2])

In [ ]:
# def test_image_height_equality(path, time, ident):
#     try:
#         after = Sample(path, time, ident)
#     except FileNotFoundError:
#         return None

#     if not np.all(np.argwhere(np.isnan(sample.height)) == np.argwhere(sample.image[:, :, 0] == 0)):
#         return ident

In [ ]:
def test_sample_shape_equality(path: Path, sample_id: str) -> tuple | None:
    """
    Check if all before/after image/height dimensions are consistent.

    For proper alignment during template matching, all four data sources
    (before image, before height, after image, after height) should have
    consistent dimensions.

    Args:
        path: Root path to data directory
        sample_id: Sample identifier

    Returns:
        Tuple of (sample_id, before_image_shape, after_image_shape,
                 before_height_shape, after_height_shape) if shapes don't match
        None if all shapes are equal
    """
    try:
        after = Sample(path, "after", sample_id)
    except FileNotFoundError:
        return None

    before = Sample(path, "before", sample_id)

    # Verify all four data sources have same dimensions
    if not (before.image.shape[:2] == before.height.shape[:2] ==
            after.image.shape[:2] == after.height.shape[:2]):
        return (sample_id, before.image.shape[:2], after.image.shape[:2],
                before.height.shape[:2], after.height.shape[:2])

In [ ]:
def find_unmatched_files(path: Path, sample_id: str) -> str | None:
    """
    Find samples missing either before or after measurements.

    For proper before/after comparison, each sample must have both a before
    and an after measurement. This function identifies samples with incomplete pairs.

    Args:
        path: Root path to data directory
        sample_id: Sample identifier

    Returns:
        sample_id if sample has missing before or after data
        None if both before and after data are present
    """
    try:
        Sample(path, "before", sample_id)
        Sample(path, "after", sample_id)
    except FileNotFoundError:
        return sample_id

In [ ]:
# Set up paths and parallel processing configuration
path = Path("../rawdata/processed")
time = "before"

# Get list of all before samples
sample_ids = [i.replace(".png", "") for i in os.listdir(path / time / "profilometer")]

# Use all available CPU cores for parallel processing
ncpu = os.cpu_count()

# Load circle detection results for later reference (from circle-fitting.ipynb)
circle_df = pd.read_csv(
    path / time / "sample-masking/circle-list/circle-list.csv",
    index_col=0
)

# Data Quality Checks

## Check 1: Unmatched Before/After Pairs

This check identifies samples that have a "before" measurement but no corresponding "after" measurement (or vice versa). Such incomplete pairs cannot be used for corrosion analysis.

In [ ]:
"""
Check for missing before/after pairs using parallel processing
"""
# Get all before and after sample IDs
before_sample_ids = [i.replace(".png", "") for i in os.listdir(path / "before" / "profilometer")]
after_sample_ids = [i.replace(".png", "") for i in os.listdir(path / "after" / "profilometer")]

# Find samples missing after data and samples missing before data
unmatched_before = Parallel(n_jobs=ncpu)(
    delayed(find_unmatched_files)(path, ident) for ident in before_sample_ids
)
unmatched_after = Parallel(n_jobs=ncpu)(
    delayed(find_unmatched_files)(path, ident) for ident in after_sample_ids
)

# Filter out None results and sort
unmatched_before = [x for x in unmatched_before if x is not None]
unmatched_after = [x for x in unmatched_after if x is not None]

unmatched_before.sort()
unmatched_after.sort()

# Display results
print("Samples with 'before' but no 'after' measurement:")
print(unmatched_before)
print("\nSamples with 'after' but no 'before' measurement:")
print(unmatched_after)

(['2-3-11-1',
  '2-3-11-2',
  '2-3-11-3',
  '2-3-11-4',
  '4-3-10-1',
  '4-3-10-2',
  '4-3-10-3',
  '4-3-10-4',
  '4-3-11-1',
  '4-3-11-2',
  '4-3-11-3',
  '4-3-11-4',
  '4-3-12-1',
  '4-3-12-2',
  '4-3-12-3',
  '4-3-12-4',
  '4-3-7-1',
  '4-3-7-2',
  '4-3-7-3',
  '4-3-7-4',
  '4-3-8-1',
  '4-3-8-2',
  '4-3-8-3',
  '4-3-8-4',
  '4-3-9-1',
  '4-3-9-2',
  '4-3-9-3',
  '4-3-9-4'],
 ['3-1-10-1',
  '3-1-10-2',
  '3-1-10-3',
  '3-1-10-4',
  '3-1-11-1',
  '3-1-11-2',
  '3-1-11-3',
  '3-1-11-4',
  '3-1-12-1',
  '3-1-12-2',
  '3-1-12-3',
  '3-1-12-4',
  '3-1-7-1',
  '3-1-7-2',
  '3-1-7-3',
  '3-1-7-4',
  '3-1-8-1',
  '3-1-8-2',
  '3-1-8-3',
  '3-1-8-4',
  '3-1-9-1',
  '3-1-9-2',
  '3-1-9-3',
  '3-1-9-4',
  '3-2-1-1',
  '3-2-1-2',
  '3-2-1-3',
  '3-2-1-4',
  '3-2-10-1',
  '3-2-10-2',
  '3-2-10-3',
  '3-2-10-4',
  '3-2-11-1',
  '3-2-11-2',
  '3-2-11-3',
  '3-2-11-4',
  '3-2-12-1',
  '3-2-12-2',
  '3-2-12-3',
  '3-2-12-4',
  '3-2-2-1',
  '3-2-2-2',
  '3-2-2-3',
  '3-2-2-4',
  '3-2-3-1',
  '3-2-3-2

# Check 2: Image vs Height Profile Shape Consistency

For each sample, the RGB image and height profile should have the same spatial dimensions. Mismatches indicate potential alignment issues between the two measurement modalities.

In [ ]:
"""
Check for image/height profile dimension mismatches in before samples
"""
before_sample_ids = [i.replace(".png", "") for i in os.listdir(path / "before" / "profilometer")]

# Test each sample for dimension consistency
image_height_inequality = Parallel(n_jobs=ncpu)(
    delayed(test_image_height_equality)(path, ident) for ident in before_sample_ids
)

# Filter None results and sort by sample ID
image_height_inequality = [x for x in image_height_inequality if x is not None]
image_height_inequality.sort()

# Display results with detailed shape information
print("Samples with image/height dimension mismatches:")
for result in image_height_inequality:
    sample_id, before_img_shape, before_height_shape, after_img_shape, after_height_shape = result
    print(f"\n{sample_id}:")
    print(f"  Before - Image: {before_img_shape}, Height: {before_height_shape}")
    print(f"  After  - Image: {after_img_shape}, Height: {after_height_shape}")

[('4-1-6-4', (1666, 1892), (1667, 1893), (1667, 1893), (1667, 1893)),
 ('4-3-1-1', (1666, 1892), (1667, 1893), (1667, 1893), (1667, 1893)),
 ('4-3-1-2', (1666, 1892), (1667, 1893), (1667, 1893), (1667, 1893)),
 ('4-3-1-3', (1666, 1892), (1667, 1894), (1667, 1893), (1667, 1893)),
 ('4-3-1-4', (1666, 1892), (1667, 1893), (1667, 1893), (1667, 1893))]

# Interpretation

When the height map shape does not agree with the image shape, the data may be shifted or corrupted. This could cause misalignment issues during centering and masking operations. Affected samples should be reviewed manually and may need to be excluded from the dataset.

# Check 3: Full Dataset Consistency

This check verifies that all four data sources (before image, before height, after image, after height) have consistent dimensions for the same sample. Inconsistency across all dimensions indicates severe data corruption.

In [ ]:
"""
Check for full dataset consistency across all four data sources
"""
before_sample_ids = [i.replace(".png", "") for i in os.listdir(path / "before" / "profilometer")]

# Test each sample for overall consistency
sample_shape_inequality = Parallel(n_jobs=ncpu)(
    delayed(test_sample_shape_equality)(path, ident) for ident in before_sample_ids
)

# Filter None results and sort by sample ID
sample_shape_inequality = [x for x in sample_shape_inequality if x is not None]
sample_shape_inequality.sort()

# Display results with detailed troubleshooting info
print("Samples with before/after shape inconsistencies:")
for result in sample_shape_inequality:
    sample_id, before_img_shape, after_img_shape, before_height_shape, after_height_shape = result
    print(f"\n{sample_id}:")
    print(f"  Before - Image: {before_img_shape}, Height: {before_height_shape}")
    print(f"  After  - Image: {after_img_shape}, Height: {after_height_shape}")
    print(f"  → Consider removing from dataset if inconsistencies cannot be resolved")

[('1-1-11-1', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-11-2', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-11-4', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-12-2', (1667, 1601), (1667, 1602), (1667, 1601), (1667, 1602)),
 ('1-1-2-1', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-2-2', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-2-3', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-2-4', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-5-1', (1666, 1601), (1667, 1601), (1666, 1601), (1667, 1601)),
 ('1-1-5-2', (1667, 1601), (1668, 1601), (1667, 1601), (1668, 1601)),
 ('1-1-5-3', (1667, 1601), (1668, 1601), (1667, 1601), (1668, 1601)),
 ('1-1-5-4', (1666, 1601), (1667, 1601), (1666, 1601), (1667, 1601)),
 ('1-1-9-1', (1667, 1601), (1668, 1601), (1667, 1601), (1668, 1601)),
 ('1-1-9-2', (1667, 1601), (1668, 1601), (1667, 1601), (1668, 1601)),
 ('1-1-9-3', (16